# 🔄 Assignment 3 — Q3: CycleGAN (Sketch ↔ Photo)
**Unpaired Image-to-Image Translation via Cycle Consistency**

Platform: Kaggle | GPU T4 x2 | PyTorch

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────────────
!pip install opendatasets gradio torch torchvision matplotlib numpy pillow scipy piqa -q
print("✅ All packages installed")


In [ ]:
# ── Cell 2: Imports ─────────────────────────────────────────────────────────
import os, time, random, itertools
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torch.cuda.amp import autocast, GradScaler
from piqa import SSIM, PSNR

import opendatasets as od
import warnings; warnings.filterwarnings('ignore')

torch.manual_seed(42); np.random.seed(42); random.seed(42)

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPU = torch.cuda.device_count()
print(f"🔥 Device: {DEVICE}  |  GPUs: {NUM_GPU}")


In [ ]:
# ── Cell 3: Download Datasets ───────────────────────────────────────────────
# TU-Berlin (HuggingFace)
!pip install datasets -q
from datasets import load_dataset
tu_berlin = load_dataset("sdiaeyu6n/tu-berlin", split="train", streaming=True)
print("✅ TU-Berlin stream ready")

# Sketchy + QuickDraw from Kaggle
od.download("https://www.kaggle.com/datasets/sharanyasundar/sketchy-dataset")
# QuickDraw is very large — we use a subset via API or local subset
# od.download("https://www.kaggle.com/c/quickdraw-doodle-recognition/data")
print("✅ Sketchy downloaded")


In [ ]:
# ── Cell 4: Configuration ───────────────────────────────────────────────────
class CFG:
    # Paths
    SKETCHY_DIR  = "sketchy-dataset"
    IMAGE_SIZE   = 128          # 128×128 for CycleGAN (memory-efficient)
    CHANNELS     = 3
    NUM_RES      = 6            # ResNet blocks (6 for Kaggle, 9 for full)

    # Training
    EPOCHS       = 40
    BATCH_SIZE   = 4            # CycleGAN is memory-hungry
    LR           = 0.0002
    BETA1        = 0.5
    BETA2        = 0.999
    LAMBDA_CYC   = 10           # cycle-consistency weight
    LAMBDA_ID    = 5            # identity loss weight
    BUFFER_SIZE  = 50           # image replay buffer

    NUM_WORKERS  = 4
    SUBSET_SIZE  = 1500         # per domain
    DEVICE       = DEVICE
    NUM_GPU      = NUM_GPU

print("⚙️  CycleGAN Config loaded")


In [ ]:
# ── Cell 5: Unpaired Dataset ────────────────────────────────────────────────
class UnpairedDataset(Dataset):
    EXTS = ('.png','.jpg','.jpeg')

    def __init__(self, root, image_size=128, augment=True, max_samples=None):
        self.size    = image_size
        self.augment = augment
        self.files   = []
        for dirpath, _, fnames in os.walk(root):
            for fn in fnames:
                if fn.lower().endswith(self.EXTS):
                    self.files.append(os.path.join(dirpath, fn))
        if max_samples:
            self.files = self.files[:max_samples]
        random.shuffle(self.files)
        print(f"  UnpairedDataset: {len(self.files)} images from '{root}'")

    def _tf(self, img):
        img = img.resize((self.size, self.size), Image.BICUBIC)
        if self.augment and random.random() > 0.5:
            img = TF.hflip(img)
        t = T.ToTensor()(img)
        return T.Normalize([0.5]*3, [0.5]*3)(t)

    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        for _ in range(5):
            try:
                return self._tf(Image.open(self.files[idx]).convert('RGB'))
            except:
                idx = (idx+1) % len(self.files)
        return torch.zeros(3, self.size, self.size)


# ── Build sketch / photo domains ────────────────────────────────────────────
# Domain A: Sketch  ─ use Sketchy sketches subfolder
# Domain B: Photo   ─ use Sketchy reference photos subfolder

def find_subdir(root, name_hint):
    """Walk root to find the first dir whose name contains name_hint."""
    for dirpath, dirs, _ in os.walk(root):
        for d in dirs:
            if name_hint.lower() in d.lower():
                return os.path.join(dirpath, d)
    return root   # fallback

sketch_root = find_subdir(CFG.SKETCHY_DIR, 'sketch')
photo_root  = find_subdir(CFG.SKETCHY_DIR, 'photo')

print("Sketch root:", sketch_root)
print("Photo  root:", photo_root)

ds_A = UnpairedDataset(sketch_root, CFG.IMAGE_SIZE, max_samples=CFG.SUBSET_SIZE)
ds_B = UnpairedDataset(photo_root,  CFG.IMAGE_SIZE, max_samples=CFG.SUBSET_SIZE)

loader_A = DataLoader(ds_A, CFG.BATCH_SIZE, shuffle=True,
                      num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
loader_B = DataLoader(ds_B, CFG.BATCH_SIZE, shuffle=True,
                      num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)

print(f"\n✅ Domain A (Sketch): {len(loader_A)} batches")
print(f"   Domain B (Photo) : {len(loader_B)} batches")


In [ ]:
# ── Cell 6: ResNet Generator ────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(ch, ch, 3), nn.InstanceNorm2d(ch), nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(ch, ch, 3), nn.InstanceNorm2d(ch)
        )
    def forward(self, x): return x + self.block(x)


class ResNetGenerator(nn.Module):
    """ResNet-based Generator for CycleGAN."""
    def __init__(self, in_ch=3, out_ch=3, nf=64, n_res=6):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_ch, nf, 7), nn.InstanceNorm2d(nf), nn.ReLU(True),
            # Downsampling
            nn.Conv2d(nf,    nf*2, 3, stride=2, padding=1),
            nn.InstanceNorm2d(nf*2), nn.ReLU(True),
            nn.Conv2d(nf*2,  nf*4, 3, stride=2, padding=1),
            nn.InstanceNorm2d(nf*4), nn.ReLU(True),
        ]
        # ResNet blocks
        for _ in range(n_res):
            layers.append(ResidualBlock(nf*4))
        # Upsampling
        layers += [
            nn.ConvTranspose2d(nf*4, nf*2, 3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(nf*2), nn.ReLU(True),
            nn.ConvTranspose2d(nf*2, nf,   3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(nf),   nn.ReLU(True),
            nn.ReflectionPad2d(3),
            nn.Conv2d(nf, out_ch, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x): return self.model(x)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

tmp = ResNetGenerator(n_res=CFG.NUM_RES)
print(f"ResNet Generator params: {count_params(tmp):,}")
del tmp


In [ ]:
# ── Cell 7: PatchGAN Discriminator ──────────────────────────────────────────
class CyclePatchGAN(nn.Module):
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, ndf,    4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf,   ndf*2,  4, 2, 1), nn.InstanceNorm2d(ndf*2),  nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf*2, ndf*4,  4, 2, 1), nn.InstanceNorm2d(ndf*4),  nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf*4, ndf*8,  4, 1, 1), nn.InstanceNorm2d(ndf*8),  nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf*8, 1,      4, 1, 1)
        )
    def forward(self, x): return self.net(x)


# ── Image Replay Buffer ──────────────────────────────────────────────────────
class ReplayBuffer:
    """Store past fakes to stabilise discriminator training."""
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, data):
        result = []
        for element in data.data:
            element = element.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(element)
                result.append(element)
            else:
                if random.random() > 0.5:
                    i = random.randint(0, self.max_size-1)
                    result.append(self.data[i].clone())
                    self.data[i] = element
                else:
                    result.append(element)
        return torch.cat(result).to(DEVICE)


print("✅ CyclePatchGAN & ReplayBuffer ready")


In [ ]:
# ── Cell 8: Weight Init & Losses ────────────────────────────────────────────
def weights_init(m):
    cls = m.__class__.__name__
    if 'Conv' in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0)
    elif 'InstanceNorm' in cls:
        if m.weight is not None: nn.init.normal_(m.weight.data, 1.0, 0.02)
        if m.bias   is not None: nn.init.constant_(m.bias.data, 0)

adv_loss = nn.MSELoss()
cyc_loss = nn.L1Loss()
ide_loss = nn.L1Loss()
print("✅ Losses: MSE (adv) + L1 (cycle + identity)")


In [ ]:
# ── Cell 9: Training Loop ────────────────────────────────────────────────────
def train_cyclegan(loader_A, loader_B, epochs=CFG.EPOCHS):
    G_AB = ResNetGenerator(CFG.CHANNELS, CFG.CHANNELS, n_res=CFG.NUM_RES).to(DEVICE)
    G_BA = ResNetGenerator(CFG.CHANNELS, CFG.CHANNELS, n_res=CFG.NUM_RES).to(DEVICE)
    D_A  = CyclePatchGAN(CFG.CHANNELS).to(DEVICE)
    D_B  = CyclePatchGAN(CFG.CHANNELS).to(DEVICE)

    if CFG.NUM_GPU > 1:
        G_AB = nn.DataParallel(G_AB); G_BA = nn.DataParallel(G_BA)
        D_A  = nn.DataParallel(D_A);  D_B  = nn.DataParallel(D_B)

    for m in [G_AB, G_BA, D_A, D_B]: m.apply(weights_init)

    # Optimisers — generators share one optimiser
    optG = optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                      lr=CFG.LR, betas=(CFG.BETA1, CFG.BETA2))
    optDA = optim.Adam(D_A.parameters(), lr=CFG.LR, betas=(CFG.BETA1, CFG.BETA2))
    optDB = optim.Adam(D_B.parameters(), lr=CFG.LR, betas=(CFG.BETA1, CFG.BETA2))

    decay = epochs // 2
    schedG  = optim.lr_scheduler.LinearLR(optG,  1.0, 0.0, total_iters=decay)
    schedDA = optim.lr_scheduler.LinearLR(optDA, 1.0, 0.0, total_iters=decay)
    schedDB = optim.lr_scheduler.LinearLR(optDB, 1.0, 0.0, total_iters=decay)

    fake_A_buf = ReplayBuffer(CFG.BUFFER_SIZE)
    fake_B_buf = ReplayBuffer(CFG.BUFFER_SIZE)
    scaler = GradScaler()

    G_log, DA_log, DB_log, cyc_log = [], [], [], []

    # Fixed samples for viz
    fix_A = next(iter(loader_A))[:4].to(DEVICE)
    fix_B = next(iter(loader_B))[:4].to(DEVICE)

    os.makedirs("checkpoints_cyc", exist_ok=True)
    os.makedirs("samples_cyc",     exist_ok=True)

    print(f"{'Ep':>4} | {'G':>7} | {'D_A':>7} | {'D_B':>7} | {'Cyc':>7} | {'Time':>6}")
    print("─" * 52)


    # ── Compute discriminator patch size once (avoid hardcoded 16×16) ──────
    with torch.no_grad():
        _probe = D_A(next(iter(loader_A)).to(DEVICE))
        _ph, _pw = _probe.shape[2], _probe.shape[3]
    del _probe

    for ep in range(1, epochs + 1):
        t0 = time.time()
        g_s = da_s = db_s = cyc_s = 0.0

        zip_loaders = zip(loader_A, loader_B)
        steps = min(len(loader_A), len(loader_B))

        for real_A, real_B in zip_loaders:
            real_A = real_A.to(DEVICE); real_B = real_B.to(DEVICE)
            bs     = real_A.size(0)
            ones   = torch.ones (bs, 1, _ph, _pw, device=DEVICE)
            zeros  = torch.zeros(bs, 1, _ph, _pw, device=DEVICE)

            # ────────────────────────────────────────────────────────
            # Generators G_AB and G_BA
            # ────────────────────────────────────────────────────────
            optG.zero_grad(set_to_none=True)
            with autocast():
                # Identity loss
                same_B = G_AB(real_B); id_B = ide_loss(same_B, real_B) * CFG.LAMBDA_ID
                same_A = G_BA(real_A); id_A = ide_loss(same_A, real_A) * CFG.LAMBDA_ID

                # Adversarial
                fake_B   = G_AB(real_A); pred_fake_B = D_B(fake_B)
                fake_A   = G_BA(real_B); pred_fake_A = D_A(fake_A)
                g_adv_B  = adv_loss(pred_fake_B, ones)
                g_adv_A  = adv_loss(pred_fake_A, ones)

                # Cycle consistency
                rec_A = G_BA(fake_B); cyc_A = cyc_loss(rec_A, real_A) * CFG.LAMBDA_CYC
                rec_B = G_AB(fake_A); cyc_B = cyc_loss(rec_B, real_B) * CFG.LAMBDA_CYC

                g_loss = g_adv_A + g_adv_B + cyc_A + cyc_B + id_A + id_B

            scaler.scale(g_loss).backward()
            scaler.step(optG)

            # ────────────────────────────────────────────────────────
            # Discriminator D_A (Sketch domain)
            # ────────────────────────────────────────────────────────
            optDA.zero_grad(set_to_none=True)
            with autocast():
                fake_A_b = fake_A_buf.push_and_pop(fake_A.detach())
                da_loss  = 0.5 * (adv_loss(D_A(real_A),   ones) +
                                   adv_loss(D_A(fake_A_b), zeros))
            scaler.scale(da_loss).backward()
            scaler.step(optDA)

            # ────────────────────────────────────────────────────────
            # Discriminator D_B (Photo domain)
            # ────────────────────────────────────────────────────────
            optDB.zero_grad(set_to_none=True)
            with autocast():
                fake_B_b = fake_B_buf.push_and_pop(fake_B.detach())
                db_loss  = 0.5 * (adv_loss(D_B(real_B),   ones) +
                                   adv_loss(D_B(fake_B_b), zeros))
            scaler.scale(db_loss).backward()
            scaler.step(optDB)

            scaler.update()

            g_s  += g_loss.item(); da_s += da_loss.item()
            db_s += db_loss.item(); cyc_s += (cyc_A + cyc_B).item()

        n = steps
        G_log.append(g_s/n); DA_log.append(da_s/n); DB_log.append(db_s/n); cyc_log.append(cyc_s/n)

        if ep > decay:
            schedG.step(); schedDA.step(); schedDB.step()

        elapsed = time.time() - t0
        print(f"{ep:>4} | {G_log[-1]:>7.4f} | {DA_log[-1]:>7.4f} | "
              f"{DB_log[-1]:>7.4f} | {cyc_log[-1]:>7.4f} | {elapsed:>5.1f}s")

        if ep % 10 == 0 or ep == epochs:
            visualize_cycle(G_AB, G_BA, fix_A, fix_B, ep)
            torch.save({
                'G_AB': G_AB.state_dict(), 'G_BA': G_BA.state_dict(),
                'D_A':  D_A.state_dict(),  'D_B':  D_B.state_dict(),
                'epoch': ep
            }, f"checkpoints_cyc/cyclegan_ep{ep:03d}.pt")

    return G_AB, G_BA, D_A, D_B, G_log, DA_log, DB_log, cyc_log


def visualize_cycle(G_AB, G_BA, real_A, real_B, ep):
    G_AB.eval(); G_BA.eval()
    with torch.no_grad():
        fake_B  = G_AB(real_A)
        rec_A   = G_BA(fake_B)
        fake_A  = G_BA(real_B)
        rec_B   = G_AB(fake_A)

    def t2np(t): return np.clip((t.cpu().permute(1,2,0).numpy()+1)/2, 0, 1)

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    fig.patch.set_facecolor('#0e1117')
    labels = [['Sketch (A)', 'Fake Photo (A→B)', 'Reconstructed A', ''],
              ['Photo (B)',  'Fake Sketch (B→A)', 'Reconstructed B', '']]

    for row, (row_imgs, lbls) in enumerate([
        ([real_A[0], fake_B[0], rec_A[0]], labels[0]),
        ([real_B[0], fake_A[0], rec_B[0]], labels[1])
    ]):
        for col, (img, lbl) in enumerate(zip(row_imgs, lbls)):
            axes[row, col].imshow(t2np(img)); axes[row, col].axis('off')
            axes[row, col].set_title(lbl, color='white', fontsize=10)
        axes[row, 3].axis('off')

    fig.suptitle(f'CycleGAN — Epoch {ep}', color='white', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'samples_cyc/epoch_{ep:03d}.png', dpi=120, bbox_inches='tight', facecolor='#0e1117')
    plt.show(); plt.close()


G_AB, G_BA, D_A, D_B, G_log, DA_log, DB_log, cyc_log = train_cyclegan(loader_A, loader_B)
print("\n✅ CycleGAN training complete")


In [ ]:
# ── Cell 10: Loss Curves ────────────────────────────────────────────────────
def plot_cyc_losses():
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.patch.set_facecolor('#0e1117')
    x = range(1, len(G_log)+1)
    configs = [
        (G_log,   '#f857a6', 'Generator Total Loss'),
        (DA_log,  '#4dd9f5', 'Discriminator A Loss'),
        (DB_log,  '#26de81', 'Discriminator B Loss'),
        (cyc_log, '#ffd166', 'Cycle Consistency Loss'),
    ]
    for ax, (log, col, title) in zip(axes, configs):
        ax.set_facecolor('#1a1d2e')
        ax.plot(x, log, color=col, lw=2.5)
        ax.fill_between(x, log, alpha=0.12, color=col)
        ax.set_title(title, color='white', fontsize=11, fontweight='bold')
        ax.set_xlabel('Epoch', color='#aaa'); ax.set_ylabel('Loss', color='#aaa')
        ax.tick_params(colors='#aaa')
        for sp in ax.spines.values(): sp.set_edgecolor('#333355')
        ax.grid(True, alpha=0.15, linestyle='--')

    fig.suptitle('CycleGAN Training Logs', color='white', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig('samples_cyc/loss_curves.png', dpi=140, bbox_inches='tight', facecolor='#0e1117')
    plt.show(); plt.close()

plot_cyc_losses()


In [ ]:
# ── Cell 11: SSIM / PSNR Evaluation ────────────────────────────────────────
ssim_metric = SSIM(n_channels=CFG.CHANNELS).to(DEVICE)
psnr_metric = PSNR().to(DEVICE)

def evaluate_cyclegan(num_batches=10):
    G_AB.eval(); G_BA.eval()
    ssim_AB, psnr_AB = [], []
    ssim_BA, psnr_BA = [], []

    with torch.no_grad():
        for i, (real_A, real_B) in enumerate(zip(loader_A, loader_B)):
            if i >= num_batches: break
            real_A, real_B = real_A.to(DEVICE), real_B.to(DEVICE)
            fake_B  = G_AB(real_A)
            rec_A   = G_BA(fake_B)
            fake_A  = G_BA(real_B)
            rec_B   = G_AB(fake_A)

            def to01(t): return (t + 1) / 2

            # Measure cycle reconstruction quality
            ssim_AB.append(ssim_metric(to01(rec_A), to01(real_A)).item())
            psnr_AB.append(psnr_metric(to01(rec_A), to01(real_A)).item())
            ssim_BA.append(ssim_metric(to01(rec_B), to01(real_B)).item())
            psnr_BA.append(psnr_metric(to01(rec_B), to01(real_B)).item())

    print("╔═══════════════════════════════════════════════╗")
    print(f"║  Cycle A→B→A   SSIM: {np.mean(ssim_AB):.4f}  PSNR: {np.mean(psnr_AB):.1f} dB  ║")
    print(f"║  Cycle B→A→B   SSIM: {np.mean(ssim_BA):.4f}  PSNR: {np.mean(psnr_BA):.1f} dB  ║")
    print("╚═══════════════════════════════════════════════╝")
    return np.mean(ssim_AB), np.mean(psnr_AB), np.mean(ssim_BA), np.mean(psnr_BA)

ssim_ab, psnr_ab, ssim_ba, psnr_ba = evaluate_cyclegan()


In [ ]:
# ── Cell 12: Gradio App ─────────────────────────────────────────────────────
import gradio as gr

pre = T.Compose([
    T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),
])

def _translate(pil_img, model):
    model.eval()
    t = pre(pil_img.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): out = model(t)[0].cpu()
    arr = np.clip((out.permute(1,2,0).numpy()+1)/2, 0, 1)
    return Image.fromarray((arr*255).astype(np.uint8))

def sketch_to_photo(img):  return _translate(img, G_AB)
def photo_to_sketch(img):  return _translate(img, G_BA)

def full_cycle(img, direction):
    if direction == "Sketch → Photo → Sketch (A→B→A)":
        step1 = _translate(img, G_AB)
        step2 = _translate(step1, G_BA)
        return step1, step2
    else:
        step1 = _translate(img, G_BA)
        step2 = _translate(step1, G_AB)
        return step1, step2

def show_random_samples():
    G_AB.eval(); G_BA.eval()
    idx_list = random.sample(range(len(ds_A)), min(5, len(ds_A)))
    outs = []
    for idx in idx_list:
        real_a = ds_A[idx].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            fake_b = G_AB(real_a)[0].cpu()
            rec_a  = G_BA(G_AB(real_a))[0].cpu()
        def t2pil(t):
            arr = np.clip((t.permute(1,2,0).numpy()+1)/2, 0, 1)
            return Image.fromarray((arr*255).astype(np.uint8))
        outs += [t2pil(real_a[0].cpu()), t2pil(fake_b), t2pil(rec_a)]
    return outs[:15]


CSS = """
body, .gradio-container {
    background: radial-gradient(ellipse at top,#0d1117 0%,#0d2137 60%,#0d1117 100%) !important;
    color:#e0e0e0 !important; font-family:'Segoe UI',sans-serif;
}
h1 { background: linear-gradient(90deg,#43e97b,#38f9d7,#4facfe);
     -webkit-background-clip:text; -webkit-text-fill-color:transparent;
     font-size:2.1rem !important; text-align:center; }
button.primary {
    background:linear-gradient(90deg,#43e97b,#38f9d7) !important;
    border:none !important; border-radius:25px !important;
    font-weight:bold !important; color:#0d1117 !important;
}
.gr-box { background:rgba(255,255,255,0.04) !important;
          border:1px solid rgba(100,220,180,0.15) !important;
          border-radius:14px !important; }
label { color:#a8e6cf !important; }
.tab-nav button.selected {
    background:linear-gradient(90deg,#43e97b,#38f9d7) !important;
    color:#0d1117 !important; border-radius:8px !important;
}
"""

with gr.Blocks(css=CSS, title="🔄 CycleGAN Studio") as demo:

    gr.HTML("<h1>🔄 CycleGAN Studio</h1>")
    gr.HTML("<p style='text-align:center;color:#888;font-size:1rem;margin-bottom:16px'>Sketch ↔ Photo Translation · Unpaired · Cycle Consistency Loss</p>")

    with gr.Tabs():

        # ──── Tab 1: Sketch → Photo ───────────────────────────────────
        with gr.Tab("✏️ Sketch → Photo"):
            gr.Markdown("### Upload a sketch and translate it to a realistic photo!")
            with gr.Row():
                with gr.Column():
                    s2p_in  = gr.Image(label="📤 Input Sketch", type='pil', height=300)
                    s2p_btn = gr.Button("🌅 Translate to Photo", variant="primary")
                with gr.Column():
                    s2p_out = gr.Image(label="🖼️ Generated Photo", height=300)
            s2p_btn.click(sketch_to_photo, s2p_in, s2p_out)

        # ──── Tab 2: Photo → Sketch ───────────────────────────────────
        with gr.Tab("📷 Photo → Sketch"):
            gr.Markdown("### Upload a photo and translate it to a sketch!")
            with gr.Row():
                with gr.Column():
                    p2s_in  = gr.Image(label="📤 Input Photo", type='pil', height=300)
                    p2s_btn = gr.Button("✏️ Convert to Sketch", variant="primary")
                with gr.Column():
                    p2s_out = gr.Image(label="🖼️ Generated Sketch", height=300)
            p2s_btn.click(photo_to_sketch, p2s_in, p2s_out)

        # ──── Tab 3: Full Cycle ────────────────────────────────────────
        with gr.Tab("🔄 Full Cycle"):
            gr.Markdown("### Demonstrate cycle consistency: A→B→A or B→A→B")
            with gr.Row():
                with gr.Column(scale=1):
                    cyc_in  = gr.Image(label="📤 Input Image", type='pil', height=250)
                    cyc_dir = gr.Dropdown(
                        ["Sketch → Photo → Sketch (A→B→A)",
                         "Photo → Sketch → Photo (B→A→B)"],
                        value="Sketch → Photo → Sketch (A→B→A)",
                        label="Cycle Direction"
                    )
                    cyc_btn = gr.Button("🔄 Run Full Cycle", variant="primary")
                with gr.Column(scale=1):
                    cyc_mid = gr.Image(label="🔀 Step 1 Translation", height=250)
                with gr.Column(scale=1):
                    cyc_rec = gr.Image(label="🔁 Step 2 Reconstruction", height=250)
            cyc_btn.click(full_cycle, [cyc_in, cyc_dir], [cyc_mid, cyc_rec])

        # ──── Tab 4: Gallery ──────────────────────────────────────────
        with gr.Tab("🖼️ Random Gallery"):
            gr.Markdown("### 5 random sketches: Input | Translated Photo | Reconstructed Sketch")
            gal_btn = gr.Button("🔀 Load Random Samples", variant="primary")
            with gr.Row():
                g_ins  = [gr.Image(label=f"Sketch {i+1}",    height=180) for i in range(5)]
            with gr.Row():
                g_fake = [gr.Image(label=f"Fake Photo {i+1}", height=180) for i in range(5)]
            with gr.Row():
                g_rec  = [gr.Image(label=f"Reconstructed {i+1}", height=180) for i in range(5)]

            def load_gal():
                outs = show_random_samples()
                while len(outs) < 15: outs.append(None)
                return outs[:15]

            gal_btn.click(load_gal, [], g_ins + g_fake + g_rec)

        # ──── Tab 5: Metrics ──────────────────────────────────────────
        with gr.Tab("📊 Metrics"):
            gr.Markdown(f"""
### CycleGAN Quantitative Results

| | SSIM | PSNR |
|---|---|---|
| **Cycle A→B→A** | {ssim_ab:.4f} | {psnr_ab:.2f} dB |
| **Cycle B→A→B** | {ssim_ba:.4f} | {psnr_ba:.2f} dB |

### Architecture Summary
- **G_AB / G_BA:** ResNet-{CFG.NUM_RES} Generator (128×128)
- **D_A / D_B:** PatchGAN Discriminator
- **Cycle Loss:** λ = {CFG.LAMBDA_CYC}  |  **Identity Loss:** λ = {CFG.LAMBDA_ID}
- **Replay Buffer:** {CFG.BUFFER_SIZE} images  |  **Batch:** {CFG.BATCH_SIZE}

### Loss Decomposition
`L_total = L_adv(G_AB) + L_adv(G_BA) + λ_cyc·(L_cyc_A + L_cyc_B) + λ_id·(L_id_A + L_id_B)`
""")

        # ──── Tab 6: About ─────────────────────────────────────────────
        with gr.Tab("📖 About"):
            gr.Markdown("""
## CycleGAN Overview
CycleGAN learns **unpaired** translation between domains using **two generators and two discriminators**.

**Key Innovations:**
- No paired data required — learns from unpaired collections
- **Cycle Consistency**: A→B→A should reconstruct A (and B→A→B should reconstruct B)
- **Identity Loss**: G_AB(B) ≈ B encourages colour preservation
- **Replay Buffer**: reduces oscillation by showing past fakes to discriminators

**vs Pix2Pix:**
| | CycleGAN | Pix2Pix |
|---|---|---|
| Data | Unpaired | Paired |
| Loss | Adversarial + Cycle + Identity | Adversarial + L1 |
| Generators | 2 (G_AB, G_BA) | 1 |
| Discriminators | 2 (D_A, D_B) | 1 |
""")

demo.launch(share=True, debug=False)
